# Detecção de Fraudes em Transações de Cartão de Crédito

Projeto prático de Ciência de Dados e Machine Learning para identificar transações fraudulentas em um dataset real e altamente desbalanceado de cartões de crédito.

**Objetivo:** construir e comparar diferentes modelos de classificação capazes de distinguir transações normais de transações fraudulentas, dando atenção especial às métricas corretas para problemas desbalanceados (recall, precisão e F1-score) e à explicabilidade do modelo final.

**Dataset:** transações de cartões de crédito europeus realizadas em setembro de 2013. As variáveis `V1` a `V28` já vêm transformadas por PCA (por questões de confidencialidade), além de `Time` (tempo desde a primeira transação do conjunto), `Amount` (valor da transação) e `Class` (0 = transação normal, 1 = fraude).

> Projeto desenvolvido a partir do conteúdo da trilha de Análise de Dados / Detecção de Anomalias da DIO, com ajustes, testes extras e conclusões próprias.


## 1. Importação das bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, recall_score,
    precision_score, f1_score
)

from xgboost import XGBClassifier
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler
import shap

import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
%matplotlib inline


## 2. Coleta dos dados

O primeiro passo de qualquer projeto de análise de dados é a coleta. Aqui usamos o dataset público de detecção de fraude em cartões de crédito (Kaggle / ULB Machine Learning Group), carregado diretamente via `pd.read_csv`.

> Se o link abaixo estiver indisponível no momento em que você executar, baixe o arquivo `creditcard.csv` direto da [página oficial no Kaggle](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) e carregue localmente com `pd.read_csv('creditcard.csv')`.


In [ ]:
url = "https://raw.githubusercontent.com/nsethi31/Kaggle-Data-Credit-Card-Fraud-Detection/master/creditcard.csv"
df = pd.read_csv(url)
df.head()


## 3. Entendendo a estrutura dos dados

In [ ]:
df.info()


In [ ]:
df.describe()


Cada linha do dataset representa uma transação financeira. Os campos são:

- **Time**: tempo (em segundos) desde a primeira transação registrada — não é uma data, e sim uma medida de decorrência temporal.
- **V1 a V28**: variáveis numéricas já transformadas por PCA para preservar a privacidade dos dados originais. Não sabemos o significado exato de cada uma, mas elas carregam padrões relevantes sobre o comportamento da transação.
- **Amount**: valor da transação.
- **Class**: variável alvo — `0` para transação normal e `1` para transação fraudulenta.


## 4. O desbalanceamento das classes

In [ ]:
class_counts = df["Class"].value_counts(normalize=True) * 100
print(class_counts)


In [ ]:
plt.figure(figsize=(5, 4))
sns.countplot(x="Class", data=df)
plt.yscale("log")
plt.title("Distribuição das classes (escala log)")
plt.xlabel("Classe (0 = normal, 1 = fraude)")
plt.ylabel("Quantidade de transações (log)")
plt.show()


O dataset é extremamente desbalanceado: menos de 0,2% das transações são fraudes. Isso é o ponto mais importante do projeto, porque muda completamente como avaliamos o modelo.

Um modelo "burro" que sempre responde "não é fraude" já acertaria mais de 99% das vezes — e seria completamente inútil. Por isso, **acurácia não é uma boa métrica aqui**. As métricas que realmente importam são:

- **Recall**: de todas as fraudes reais, quantas o modelo conseguiu identificar.
- **Precisão (precision)**: das transações que o modelo marcou como fraude, quantas realmente eram fraude.
- **F1-score**: a média harmônica entre recall e precisão, dando uma visão de equilíbrio entre as duas.

Em detecção de fraude, normalmente priorizamos o **recall** — é mais caro deixar uma fraude passar do que investigar um falso positivo.


## 5. Engenharia de atributos (Feature Engineering)

As colunas originais nem sempre estão na melhor forma para o modelo aprender. Aqui aplicamos duas transformações simples e muito usadas em dados financeiros:

1. **Transformação logarítmica** (`log1p`) sobre `Amount`, para reduzir o impacto de valores muito grandes e deixar a distribuição mais equilibrada (usamos `log1p` em vez de `log` para evitar erro quando o valor é zero).
2. **Padronização** (`StandardScaler`) sobre `Amount`, para que a variável passe a ter média 0 e desvio padrão 1 — muitos modelos (como a regressão logística) são sensíveis à escala dos dados.


In [ ]:
df["Amount_log"] = np.log1p(df["Amount"])

scaler = StandardScaler()
df["Amount_scaled"] = scaler.fit_transform(df[["Amount"]])

df[["Amount", "Amount_log", "Amount_scaled"]].head()


## 6. Separação em treino e teste

Separamos as features (`X`) do alvo (`y`) e dividimos em 70% treino / 30% teste. O parâmetro `stratify=y` é essencial aqui: como o dataset é desbalanceado, ele garante que a proporção de fraudes seja mantida tanto no treino quanto no teste.


In [ ]:
X = df.drop("Class", axis=1)
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Treino: {X_train.shape[0]} amostras ({y_train.sum()} fraudes)")
print(f"Teste:  {X_test.shape[0]} amostras ({y_test.sum()} fraudes)")


## 7. Modelo baseline: Regressão Logística

Começamos com o modelo mais simples, usado como referência (baseline) para comparar os modelos seguintes.


In [ ]:
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)

y_pred_lr = log_reg.predict(X_test)

print(classification_report(y_test, y_pred_lr, digits=3))


In [ ]:
cm = confusion_matrix(y_test, y_pred_lr)
plt.figure(figsize=(4, 3))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Matriz de confusão — Regressão Logística")
plt.xlabel("Previsto")
plt.ylabel("Real")
plt.show()


## 8. Curva ROC e curva Precision-Recall

A **curva ROC** mostra a relação entre taxa de falsos positivos e recall (taxa de verdadeiros positivos) em diferentes limiares de decisão — quanto mais próxima do canto superior esquerdo, melhor o modelo.

Em problemas desbalanceados como este, porém, a **curva Precision-Recall** costuma ser mais informativa, já que ela foca diretamente no comportamento da classe minoritária (a fraude).


In [ ]:
y_proba_lr = log_reg.predict_proba(X_test)[:, 1]

fpr, tpr, _ = roc_curve(y_test, y_proba_lr)
auc_lr = roc_auc_score(y_test, y_proba_lr)

plt.figure(figsize=(5, 4))
plt.plot(fpr, tpr, label=f"AUC = {auc_lr:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("Taxa de falsos positivos")
plt.ylabel("Recall (verdadeiros positivos)")
plt.title("Curva ROC — Regressão Logística")
plt.legend()
plt.show()


In [ ]:
precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_proba_lr)

plt.figure(figsize=(5, 4))
plt.plot(recall_vals, precision_vals)
plt.xlabel("Recall")
plt.ylabel("Precisão")
plt.title("Curva Precision-Recall — Regressão Logística")
plt.show()


## 9. Técnicas de balanceamento: Undersampling vs Oversampling

Existem duas estratégias clássicas para lidar com o desbalanceamento:

- **Undersampling**: reduz a classe majoritária (transações normais) até ficar do mesmo tamanho da classe minoritária. Perde-se informação, mas o treino fica mais rápido e equilibrado.
- **Oversampling**: gera novos exemplos sintéticos da classe minoritária (fraude). Não perde dados, mas pode introduzir ruído.

Como teste extra, comparamos o impacto das duas técnicas em uma regressão logística simples, treinada apenas com o conjunto de treino balanceado (a avaliação continua sendo feita no conjunto de teste original, sem balancear).


In [ ]:
under = RandomUnderSampler(random_state=42)
X_train_under, y_train_under = under.fit_resample(X_train, y_train)

over = RandomOverSampler(random_state=42)
X_train_over, y_train_over = over.fit_resample(X_train, y_train)

resultados_balanceamento = {}

for nome, (X_bal, y_bal) in {
    "Undersampling": (X_train_under, y_train_under),
    "Oversampling": (X_train_over, y_train_over),
}.items():
    modelo = LogisticRegression(max_iter=1000)
    modelo.fit(X_bal, y_bal)
    pred = modelo.predict(X_test)
    resultados_balanceamento[nome] = {
        "recall": recall_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "f1": f1_score(y_test, pred),
    }

pd.DataFrame(resultados_balanceamento).T


Na prática, o undersampling tende a aumentar bastante o recall (o modelo passa a "ver" muito mais fraudes proporcionalmente), mas geralmente à custa de mais falsos positivos. O oversampling costuma ficar em um meio-termo. Vale comparar esses números com os modelos seguintes.


## 10. Modelo 2: Random Forest

O Random Forest é um conjunto (*ensemble*) de várias árvores de decisão independentes, cujas previsões são combinadas. Usamos `class_weight="balanced"` para que o próprio modelo dê mais peso à classe minoritária durante o treino, sem precisar balancear manualmente os dados.


In [ ]:
rf = RandomForestClassifier(
    n_estimators=50,
    max_depth=10,
    class_weight="balanced",
    random_state=42
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
print(classification_report(y_test, y_pred_rf, digits=3))


## 11. Pipeline e ajuste de threshold

Um **Pipeline** organiza várias etapas de processamento (padronização + modelo) em um único fluxo, garantindo que treino, teste e produção sigam exatamente os mesmos passos.

Por padrão, classificamos como fraude quando a probabilidade prevista é maior que **0,5**. Reduzindo esse limiar (threshold) para **0,3**, o modelo passa a marcar mais transações como fraude — aumentando o recall, ao custo de alguma precisão.


In [ ]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000))
])
pipe.fit(X_train, y_train)

proba_pipe = pipe.predict_proba(X_test)[:, 1]

for threshold in [0.5, 0.3]:
    pred_thresh = (proba_pipe >= threshold).astype(int)
    print(f"Threshold = {threshold}")
    print(f"  Recall:    {recall_score(y_test, pred_thresh):.3f}")
    print(f"  Precisão:  {precision_score(y_test, pred_thresh):.3f}")
    print(f"  F1-score:  {f1_score(y_test, pred_thresh):.3f}")
    print()


## 12. Modelo 3: XGBoost

O XGBoost é baseado em *boosting*: diferente do Random Forest (onde as árvores são treinadas de forma independente), aqui as árvores são treinadas em sequência, e cada nova árvore tenta corrigir os erros das anteriores. É um dos algoritmos mais usados em competições e em aplicações reais de detecção de fraude e anomalias.


In [ ]:
xgb = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    eval_metric="logloss",
    random_state=42
)
xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)
print(classification_report(y_test, y_pred_xgb, digits=3))


## 13. Importância das variáveis

Esse gráfico mostra o peso que cada variável teve nas decisões do modelo XGBoost. Quanto maior a barra, maior a influência da variável na classificação de uma transação como fraude.


In [ ]:
importancias = pd.Series(xgb.feature_importances_, index=X_train.columns)
importancias = importancias.sort_values(ascending=False).head(15)

plt.figure(figsize=(7, 5))
sns.barplot(x=importancias.values, y=importancias.index)
plt.title("Importância das variáveis — XGBoost")
plt.xlabel("Importância")
plt.show()


## 14. Ajuste de hiperparâmetros (GridSearchCV)

Hiperparâmetros (como profundidade máxima das árvores ou número de estimadores) não são aprendidos pelo modelo — precisamos defini-los. O `GridSearchCV` testa automaticamente várias combinações e escolhe a melhor com base em uma métrica escolhida — aqui, o **recall**, já que esse é o nosso foco principal em detecção de fraude.


In [ ]:
param_grid = {
    "max_depth": [3, 5],
    "n_estimators": [50, 100],
}

grid_search = GridSearchCV(
    XGBClassifier(eval_metric="logloss", random_state=42),
    param_grid,
    scoring="recall",
    cv=3
)
grid_search.fit(X_train, y_train)

print("Melhores parâmetros:", grid_search.best_params_)

melhor_modelo = grid_search.best_estimator_
y_pred_best = melhor_modelo.predict(X_test)
print(classification_report(y_test, y_pred_best, digits=3))


## 15. Explicabilidade com SHAP

Importância de variáveis mostra o que é relevante *no geral*. O **SHAP** vai além: ele explica o impacto de cada variável em *cada previsão individual*. Isso é essencial em fraude, onde muitas vezes é preciso justificar por que uma transação específica foi marcada como suspeita — questão de confiança, auditoria e uso responsável em produção.


In [ ]:
explainer = shap.TreeExplainer(melhor_modelo)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test)


## 16. Comparação final dos modelos

Reunindo as métricas (na classe fraude) de todos os modelos testados, para decidir qual abordagem teve o melhor desempenho neste projeto.


In [ ]:
comparacao = {
    "Regressão Logística (baseline)": {
        "recall": recall_score(y_test, y_pred_lr),
        "precision": precision_score(y_test, y_pred_lr),
        "f1": f1_score(y_test, y_pred_lr),
        "auc": roc_auc_score(y_test, y_proba_lr),
    },
    "Random Forest": {
        "recall": recall_score(y_test, y_pred_rf),
        "precision": precision_score(y_test, y_pred_rf),
        "f1": f1_score(y_test, y_pred_rf),
        "auc": roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1]),
    },
    "XGBoost": {
        "recall": recall_score(y_test, y_pred_xgb),
        "precision": precision_score(y_test, y_pred_xgb),
        "f1": f1_score(y_test, y_pred_xgb),
        "auc": roc_auc_score(y_test, xgb.predict_proba(X_test)[:, 1]),
    },
    "XGBoost + GridSearchCV": {
        "recall": recall_score(y_test, y_pred_best),
        "precision": precision_score(y_test, y_pred_best),
        "f1": f1_score(y_test, y_pred_best),
        "auc": roc_auc_score(y_test, melhor_modelo.predict_proba(X_test)[:, 1]),
    },
}

df_comparacao = pd.DataFrame(comparacao).T.round(3)
df_comparacao


In [ ]:
df_comparacao[["recall", "precision", "f1"]].plot(kind="bar", figsize=(8, 5))
plt.title("Comparação de modelos — classe fraude")
plt.ylabel("Score")
plt.xticks(rotation=20)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


## 17. Conclusão

> ✍️ **Preencha esta seção com os resultados reais obtidos na sua execução** — os números variam um pouco a cada rodada por conta de aleatoriedade nos modelos.

Ao longo deste projeto:

- Vimos que a **acurácia é uma métrica enganosa** em datasets desbalanceados como este, e que **recall, precisão e F1-score** são as métricas corretas para avaliar a detecção de fraude.
- Testamos **undersampling** e **oversampling** como estratégias de balanceamento, comparando o impacto de cada uma no recall e na precisão.
- Evoluímos de um modelo simples (**Regressão Logística**) para modelos mais robustos (**Random Forest** e **XGBoost**), observando ganhos de desempenho a cada etapa.
- Vimos como o **ajuste de threshold** permite priorizar recall (capturar mais fraudes) em troca de alguma precisão — uma decisão de negócio, não só técnica.
- Usamos **GridSearchCV** para otimizar os hiperparâmetros do XGBoost com foco em recall.
- Usamos **SHAP** para entender quais variáveis (destacando-se `V4` e `V14`) mais influenciam a decisão do modelo, importante para auditoria e confiança em produção.

Na minha execução, o modelo **[preencher: ex. XGBoost + GridSearchCV]** apresentou o melhor equilíbrio entre recall e precisão, por isso seria minha escolha para um cenário real de detecção de fraudes.

**Próximos passos possíveis:** testar SMOTE (oversampling sintético) em vez de oversampling aleatório, testar outros modelos (LightGBM, redes neurais), e simular um cenário de custo de negócio (custo de uma fraude não detectada vs. custo de investigar um falso positivo) para escolher o threshold ideal.
